# Simulation Validation - Sim-Grounded Multi-Plan Replanning Pipeline

Run **all episodes** through the REFLECT pipeline using `generate_replan_multi_plan_sim`:
- **Thinking proposer** (`reasoning_effort="medium"`) generates 4 schema-grounded candidate actions per step
- **Fast scorer** (`reasoning_effort="none"`) selects the best candidate via logprob scoring (A/B/C/D/E=Done)
- Each selected action is **executed in the simulator** and the scene graph is read back before the next step
- Task success is evaluated via `check_task_success` after the full correction loop
- Checkpoints every N episodes to `checkpoint.json`; final results to `results__<model>.json`

Analysis:
1. **Co-plan success vs episode-level entropy** - does higher LLM uncertainty predict task failure?
2. **Per-episode trace navigator** - drill into every LLM decision, its alternatives, entropy, and post-action state
3. **Failed localisation vs entropy spikes** - when the LLM misidentifies the failure step, was entropy elevated?


In [ ]:
import os, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from IPython.display import display, Markdown

from reflect.core.paths import local_uncertainty_root, sim_data_root, prompts_dir
from reflect.llm.prompter import PortkeyLLMPrompter
from reflect.pipelines.fast_validation import EpisodeConfig, process_episode

plt.style.use("ggplot")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150})

In [ ]:
PORTKEY_API_KEY = os.environ["PORTKEY_API_KEY"]  # set in your shell or .env (see .env.example)
MODEL = "gpt-5.4"
GEN_REASONING_EFFORT = "medium"    # thinking proposer
SCORE_REASONING_EFFORT = "none"    # fast scorer
WITH_AUDIO = 1
NUM_CANDIDATES = 4
MAX_STEPS = 30
SIM_GROUNDED_REPLAN = True         # propose+execute each step in sim; False = plan-then-execute
CHECKPOINT_INTERVAL = 5            # write checkpoint every N episodes

from reflect.core.paths import sim_data_root

DATA_ROOT = str(sim_data_root())  # honors REFLECT_DATA_ROOT
PROMPT_TEMPLATE_PATH = prompts_dir() / "sim_prompts.json"
MAX_EPISODES = 0                   # 0 = run ALL discovered episodes

MODEL_SLUG = MODEL.replace("/", "_").replace(".", "_")
RUN_TAG = f"{MODEL_SLUG}__sim_grounded" if SIM_GROUNDED_REPLAN else MODEL_SLUG
OUTPUT_DIR = local_uncertainty_root()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.json"
RESULTS_PATH = OUTPUT_DIR / f"results__{RUN_TAG}.json"

print(f"Model       : {MODEL}  (gen={GEN_REASONING_EFFORT}, score={SCORE_REASONING_EFFORT})")
print(f"Sim-grounded: {SIM_GROUNDED_REPLAN}")
print(f"Data root   : {DATA_ROOT}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Results     : {RESULTS_PATH}")
print(f"Checkpoint  : every {CHECKPOINT_INTERVAL} episodes → {CHECKPOINT_PATH}")
print(f"Max episodes: {'all' if MAX_EPISODES == 0 else MAX_EPISODES}")


## Run all episodes

In [ ]:
with open(PROMPT_TEMPLATE_PATH) as f:
    prompt_template = json.load(f)

CACHE_DIR = os.path.join(DATA_ROOT, ".llm_cache", MODEL)
gen_prompter = PortkeyLLMPrompter(
    portkey_api_key=PORTKEY_API_KEY,
    model=MODEL,
    reasoning_effort=GEN_REASONING_EFFORT,
    cache_dir=CACHE_DIR,
)
score_prompter = PortkeyLLMPrompter(
    portkey_api_key=PORTKEY_API_KEY,
    model=MODEL,
    reasoning_effort=SCORE_REASONING_EFFORT,
    cache_dir=CACHE_DIR,
)

# Discover episodes
sim_root = Path(DATA_ROOT)
episode_list = []  # (task_name, episode_name, episode_dir_path)
for task_dir in sorted(sim_root.iterdir()):
    if not task_dir.is_dir() or task_dir.name.startswith("."):
        continue
    for ep_dir in sorted(task_dir.iterdir()):
        if ep_dir.is_dir():
            episode_list.append((task_dir.name, ep_dir.name, ep_dir))
if MAX_EPISODES > 0:
    episode_list = episode_list[:MAX_EPISODES]

print(f"LLM cache   : {CACHE_DIR}")
print(f"Discovered  : {len(episode_list)} episodes")
for t, e, _ in episode_list[:5]:
    print(f"  {t}/{e}")
if len(episode_list) > 5:
    print(f"  ... and {len(episode_list) - 5} more")

In [ ]:
all_results = {}  # "task/episode" → process_episode result dict

t_start = time.time()
for i, (task_name, episode_name, episode_dir) in enumerate(episode_list):
    key = f"{task_name}/{episode_name}"
    print(f"[{i+1:03d}/{len(episode_list)}] {key} ...", end="", flush=True)
    ep_start = time.time()

    cfg = EpisodeConfig(
        data_path=str(episode_dir),
        task_name=task_name,
        episode_name=episode_name,
        llm_prompter=score_prompter,
        prompt_template=prompt_template,
        with_audio=WITH_AUDIO,
        multi_plan_replan=True,
        sim_grounded_replan=SIM_GROUNDED_REPLAN,
        gen_prompter=gen_prompter,
    )
    result = process_episode(cfg)
    ep_elapsed = time.time() - ep_start

    all_results[key] = result
    status = result.get("status", "error")
    correction_dict = result.get("correction_dict") or {}
    replan_dict = result.get("replan_dict") or {}
    coplan_ok = (
        "✓" if correction_dict.get("success") is True
        else ("✗" if correction_dict.get("success") is False else "-")
    )
    n_steps = replan_dict.get("num_steps", "-")
    print(f" {status}  coplan={coplan_ok}  steps={n_steps}  ({ep_elapsed:.1f}s)", flush=True)

    if (i + 1) % CHECKPOINT_INTERVAL == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump(all_results, f, default=str)
        print(f"  [checkpoint] {i+1}/{len(episode_list)} → {CHECKPOINT_PATH}")

# Final save
with open(RESULTS_PATH, "w") as f:
    json.dump(all_results, f, default=str)
elapsed = time.time() - t_start
print(f"All {len(all_results)} results saved → {RESULTS_PATH}  ({elapsed:.1f}s total)")


## Re-run failed episodes from checkpoint 

Loads the checkpoint, identifies episodes with status='error', re-runs them with the fixed pipeline, and updates all_results + checkpoint/results files.

In [ ]:
with open(CHECKPOINT_PATH) as f:
    _ckpt = json.load(f)

# Merge checkpoint into all_results so we don't re-run already-successful ones
for _k, _v in _ckpt.items():
    if _k not in all_results or all_results[_k].get("status") == "error":
        all_results[_k] = _v

_failed_keys = {k for k, v in all_results.items() if v.get("status") == "error"}
_ep_index = {f"{t}/{e}": (t, e, d) for t, e, d in episode_list}

print(f"Failed episodes to retry: {len(_failed_keys)}")
for _key in sorted(_failed_keys):
    print(f"  {_key}")

_t_start = time.time()
_retried, _recovered = 0, 0
for _key in sorted(_failed_keys):
    if _key not in _ep_index:
        print(f"  [SKIP] {_key} not in episode_list")
        continue
    _task_name, _episode_name, _episode_dir = _ep_index[_key]
    print(f"  Retrying {_key} ...", end="", flush=True)
    _ep_start = time.time()
    _cfg = EpisodeConfig(
        data_path=str(_episode_dir),
        task_name=_task_name,
        episode_name=_episode_name,
        llm_prompter=score_prompter,
        prompt_template=prompt_template,
        with_audio=WITH_AUDIO,
        multi_plan_replan=True,
        sim_grounded_replan=SIM_GROUNDED_REPLAN,
        gen_prompter=gen_prompter,
    )
    _result = process_episode(_cfg)
    _elapsed = time.time() - _ep_start
    all_results[_key] = _result
    _status = _result.get("status", "error")
    _retried += 1
    if _status != "error":
        _recovered += 1
    print(f" {_status}  ({_elapsed:.1f}s)", flush=True)

# Persist updated results
with open(CHECKPOINT_PATH, "w") as f:
    json.dump(all_results, f, default=str)
with open(RESULTS_PATH, "w") as f:
    json.dump(all_results, f, default=str)

print(f"\nRetried {_retried} episodes, {_recovered} recovered → {CHECKPOINT_PATH}")


## Load results & overview

In [ ]:
# Build DataFrames from in-memory results and export CSVs for persistence
episode_rows = []
trace_rows = []
plan_trace_rows = []
timing_rows = []

for key, result in all_results.items():
    task_name, episode_name = key.split("/", 1)
    reasoning_dict = result.get("reasoning_dict") or {}
    correction_dict = result.get("correction_dict") or {}
    replan_dict_r = result.get("replan_dict") or {}
    coplan_success = (
        correction_dict.get("success")
        if result.get("status") == "ok" and result.get("replay_available")
        else None
    )
    episode_rows.append({
        "task": task_name,
        "episode": episode_name,
        "status": result.get("status"),
        "artifact_mode": result.get("artifact_mode"),
        "replay_available": result.get("replay_available", False),
        "sim_grounded": result.get("sim_grounded", SIM_GROUNDED_REPLAN),
        "coplan_success": coplan_success,
        "num_steps": replan_dict_r.get("num_steps"),
        "pred_failure_step": reasoning_dict.get("pred_failure_step"),
        "gt_failure_step": reasoning_dict.get("gt_failure_step"),
        "pred_failure_reason": reasoning_dict.get("pred_failure_reason", ""),
        "gt_failure_reason": reasoning_dict.get("gt_failure_reason", ""),
        "error_type": reasoning_dict.get("error_type", ""),
        "gt_error_type": reasoning_dict.get("gt_error_type", ""),
    })
    for trace in (reasoning_dict.get("detector_trace") or []):
        score = trace.get("score") or {}
        trace_rows.append({
            "task": task_name,
            "episode": episode_name,
            "step": trace.get("step"),
            "subgoal": trace.get("subgoal", ""),
            "evaluation_active": bool(trace.get("evaluation_active")),
            "predicted_success": trace.get("predicted_success"),
            "oracle_success": trace.get("oracle_success"),
            "predicted_label": trace.get("predicted_label"),
            "confidence": score.get("confidence"),
            "entropy": score.get("entropy"),
            "score_status": score.get("score_status", "unscored"),
        })
    for entry in (replan_dict_r.get("plan_trace") or []):
        plan_trace_rows.append({
            "task": task_name,
            "episode": episode_name,
            "step_index": entry.get("step_index"),
            "candidates_json": json.dumps(entry.get("candidates", [])),
            "selected_label": entry.get("selected_label"),
            "selected_action": json.dumps(entry.get("selected_action")),
            "curr_state_after": entry.get("curr_state_after", ""),
            "done": entry.get("done", False),
            "confidence": entry.get("confidence"),
            "entropy": entry.get("entropy"),
            "option_probs": json.dumps(entry.get("option_probs")),
            "score_status": entry.get("score_status"),
        })
    for stage_name, elapsed_s in (result.get("stage_times") or []):
        timing_rows.append({
            "task": task_name,
            "episode": episode_name,
            "stage": stage_name,
            "elapsed_s": round(elapsed_s, 3),
        })

episode_metrics = pd.DataFrame(episode_rows)
detector_trace  = pd.DataFrame(trace_rows)
plan_trace      = pd.DataFrame(plan_trace_rows)
stage_timings   = pd.DataFrame(timing_rows)

# trace_jsons used by episode navigator below
trace_jsons = all_results

# Export CSVs
ep_path = OUTPUT_DIR / f"episode_metrics__{RUN_TAG}.csv"
dt_path = OUTPUT_DIR / f"detector_trace__{RUN_TAG}.csv"
episode_metrics.to_csv(ep_path, index=False)
detector_trace.to_csv(dt_path, index=False)
if not plan_trace.empty:
    plan_trace.to_csv(OUTPUT_DIR / f"plan_trace__{RUN_TAG}.csv", index=False)
if not stage_timings.empty:
    stage_timings.to_csv(OUTPUT_DIR / f"stage_timings__{RUN_TAG}.csv", index=False)

display(Markdown(
    f"**{len(episode_metrics)}** episodes - "
    f"**{len(detector_trace)}** verifier rows - "
    f"**{len(plan_trace)}** plan-trace rows"
))


## Build per-episode entropy predictors

All downstream analyses depend on a single per-episode table.
We compute several candidate "uncertainty" predictors from the raw
per-step traces so we can compare them (mean / max / high-count / last /
proposer). Verifier entropy (subgoal detector) and proposer entropy
(plan-selection A/B/C/D/E) are kept separate because the two
`reasoning_effort` settings produce different confidence regimes.


In [ ]:
# Threshold for "high-entropy" step - chosen just above the 75th-percentile of
# verifier entropy (~1.4e-3) so that counts reflect genuinely uncertain steps.
HIGH_ENT_THRESHOLD = 0.10

def _agg_entropy(frame, prefix):
    if frame.empty:
        return pd.DataFrame(columns=["task", "episode",
                                     f"{prefix}_mean", f"{prefix}_median",
                                     f"{prefix}_max", f"{prefix}_std",
                                     f"{prefix}_high_count", f"{prefix}_n_steps",
                                     f"{prefix}_last"])
    frame = frame.dropna(subset=["entropy"]).copy()
    def per_ep(g):
        g = g.sort_values("step" if "step" in g.columns else "step_index")
        return pd.Series({
            f"{prefix}_mean":       g["entropy"].mean(),
            f"{prefix}_median":     g["entropy"].median(),
            f"{prefix}_max":        g["entropy"].max(),
            f"{prefix}_std":        g["entropy"].std(),
            f"{prefix}_high_count": int((g["entropy"] > HIGH_ENT_THRESHOLD).sum()),
            f"{prefix}_n_steps":    int(len(g)),
            f"{prefix}_last":       g["entropy"].iloc[-1],
        })
    return frame.groupby(["task", "episode"]).apply(per_ep, include_groups=False).reset_index()

verif_agg = _agg_entropy(detector_trace, "verif")
plan_agg  = _agg_entropy(plan_trace, "plan")

ep_stats = (
    episode_metrics.merge(verif_agg, on=["task", "episode"], how="left")
                   .merge(plan_agg,  on=["task", "episode"], how="left")
)

# Within-task z-scores (controls for baseline per-task entropy)
for col in ["verif_mean", "verif_max", "plan_mean", "plan_max"]:
    if col in ep_stats.columns:
        ep_stats[f"{col}_z"] = (
            ep_stats.groupby("task")[col]
                    .transform(lambda s: (s - s.mean()) / s.std(ddof=0))
        )

# Restrict to episodes with a coplan outcome and verifier entropy (analysis set)
ep_analysis = ep_stats.dropna(subset=["coplan_success", "verif_mean"]).copy()
ep_analysis["coplan_success"] = ep_analysis["coplan_success"].astype(bool)
ep_analysis["failure"] = (~ep_analysis["coplan_success"]).astype(int)

print(f"Analysis set: {len(ep_analysis)} episodes "
      f"({int(ep_analysis.coplan_success.sum())} success, "
      f"{int((~ep_analysis.coplan_success).sum())} failure)")
print(f"High-entropy threshold: {HIGH_ENT_THRESHOLD}")
ep_analysis[["task", "episode", "coplan_success",
             "verif_mean", "verif_max", "verif_high_count",
             "plan_mean", "plan_max"]].head()


## Pooled analysis: entropy → co-plan failure

The entropy distribution spans ~6 orders of magnitude (min ≈ 1e-6, max ≈ 1.0).
Linear-scale plots compress almost everything to zero - we use log-y throughout.

Three plots below:

1. **Log-scale violin** - distribution of per-episode mean verifier entropy,
   success vs failure, with medians and Mann-Whitney U test.
2. **ROC curve** - how well mean entropy, by itself, predicts co-plan failure.
   AUC = 0.5 is random; > 0.7 would be genuinely usable for gating.
3. **Calibration** - bin episodes by entropy quintile and plot observed
   failure rate. A monotone curve means entropy is well-calibrated as a
   hardness signal.


In [ ]:
# ── Pooled entropy analysis (log-scale violin + ROC + calibration) ───────────
from sklearn.metrics import roc_curve, roc_auc_score, average_precision_score

EPS = 1e-6  # floor for log scale

def _log(x):
    return np.log10(np.asarray(x) + EPS)

succ = ep_analysis.loc[ep_analysis.coplan_success, "verif_mean"].values
fail = ep_analysis.loc[~ep_analysis.coplan_success, "verif_mean"].values
u, p_mw = scipy_stats.mannwhitneyu(succ, fail, alternative="two-sided")
auc_mean = roc_auc_score(ep_analysis.failure, ep_analysis.verif_mean)
auc_max  = roc_auc_score(ep_analysis.failure, ep_analysis.verif_max.fillna(0))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (1) Log-scale violin
ax = axes[0]
parts = ax.violinplot([_log(succ), _log(fail)], showmedians=True, widths=0.8)
for i, pc in enumerate(parts["bodies"]):
    pc.set_facecolor(["#4caf50", "#e53935"][i])
    pc.set_alpha(0.55); pc.set_edgecolor("black")
ax.set_xticks([1, 2])
ax.set_xticklabels([f"Success\n(n={len(succ)})", f"Failure\n(n={len(fail)})"])
ax.set_ylabel("log10(mean verifier entropy + 1e-6)")
ax.set_title(f"Mean verifier entropy by co-plan outcome\nMann-Whitney p={p_mw:.4f}")

# (2) ROC
ax = axes[1]
fpr, tpr, _ = roc_curve(ep_analysis.failure, ep_analysis.verif_mean)
ax.plot(fpr, tpr, color="#1f77b4", lw=2, label=f"mean ent  (AUC={auc_mean:.3f})")
fpr2, tpr2, _ = roc_curve(ep_analysis.failure, ep_analysis.verif_max.fillna(0))
ax.plot(fpr2, tpr2, color="#ff7f0e", lw=2, ls="--", label=f"max ent   (AUC={auc_max:.3f})")
ax.plot([0, 1], [0, 1], color="grey", lw=1, ls=":")
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title("ROC: entropy → co-plan failure")
ax.legend(loc="lower right")

# (3) Calibration by entropy quintile
ax = axes[2]
ep_analysis_sorted = ep_analysis.copy()
ep_analysis_sorted["bin"] = pd.qcut(
    ep_analysis_sorted["verif_mean"], q=5,
    labels=False, duplicates="drop"
)
cal = (
    ep_analysis_sorted.groupby("bin")
    .agg(mean_ent=("verif_mean", "mean"),
         failure_rate=("failure", "mean"),
         n=("failure", "size"))
    .reset_index()
)
ax.plot(cal["bin"], cal["failure_rate"], "o-", color="#c0392b", lw=2, markersize=10)
for _, r in cal.iterrows():
    ax.annotate(f"n={int(r['n'])}\nent≈{r['mean_ent']:.1e}",
                (r["bin"], r["failure_rate"]),
                textcoords="offset points", xytext=(8, -5), fontsize=8)
ax.axhline(ep_analysis.failure.mean(), color="grey", ls="--", lw=1,
           label=f"base rate ({ep_analysis.failure.mean():.0%})")
ax.set_xlabel("Entropy quintile (low → high)")
ax.set_ylabel("Observed co-plan failure rate")
ax.set_ylim(0, 1)
ax.set_title("Calibration: failure rate vs entropy quintile")
ax.legend()

plt.tight_layout()
plt.show()

r_pb, p_pb = scipy_stats.pointbiserialr(ep_analysis.failure, ep_analysis.verif_mean)
print(f"Point-biserial r (mean entropy ↔ failure) = {r_pb:+.4f},  p = {p_pb:.4f}")
print(f"ROC-AUC  mean entropy → failure = {auc_mean:.4f}")
print(f"ROC-AUC  max  entropy → failure = {auc_max:.4f}")
print(f"Mann-Whitney U(mean) p = {p_mw:.4f}")


## Per-task heterogeneity

The pooled plot hides whether entropy works **uniformly** across tasks or
only in certain task families. For each task we plot the success /
failure entropy distributions side-by-side and summarise the gap
(`failure_mean − success_mean`) as a heatmap.

A useful hardness signal should have a **positive and consistent** gap
across tasks. If the gap flips sign, the signal is task-dependent and
can't be used as a universal gate.


In [ ]:
# ── Per-task entropy facets ──────────────────────────────────────────────────
tasks = sorted(ep_analysis["task"].unique())
n_tasks = len(tasks)
ncols = 5
nrows = int(np.ceil(n_tasks / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.0 * nrows),
                         sharey=True)
axes = np.array(axes).reshape(-1)

for i, t in enumerate(tasks):
    ax = axes[i]
    sub = ep_analysis[ep_analysis.task == t]
    s = sub.loc[sub.coplan_success, "verif_mean"].values
    f = sub.loc[~sub.coplan_success, "verif_mean"].values
    groups = [g for g in [s, f] if len(g)]
    if not groups:
        continue
    parts = ax.violinplot([_log(g) for g in groups], showmedians=True, widths=0.8)
    colors = []
    if len(s): colors.append("#4caf50")
    if len(f): colors.append("#e53935")
    for j, pc in enumerate(parts["bodies"]):
        pc.set_facecolor(colors[j]); pc.set_alpha(0.55); pc.set_edgecolor("black")
    labels = []
    ticks = []
    idx = 1
    if len(s): ticks.append(idx); labels.append(f"S\nn={len(s)}"); idx += 1
    if len(f): ticks.append(idx); labels.append(f"F\nn={len(f)}"); idx += 1
    ax.set_xticks(ticks); ax.set_xticklabels(labels, fontsize=8)
    ax.set_title(f"{t}  ({int(sub.coplan_success.sum())}/{len(sub)})",
                 fontsize=9)

for j in range(n_tasks, len(axes)):
    axes[j].axis("off")
for ax in axes[:n_tasks:ncols]:
    ax.set_ylabel("log10(ent)")
plt.suptitle("Per-task mean verifier entropy: Success (S) vs Failure (F)", y=1.02)
plt.tight_layout()
plt.show()

# ── Heatmap: per-task gap (fail - succ) and AUC ─────────────────────────────
rows = []
for t in tasks:
    sub = ep_analysis[ep_analysis.task == t]
    s = sub.loc[sub.coplan_success, "verif_mean"]
    f = sub.loc[~sub.coplan_success, "verif_mean"]
    try:
        task_auc = roc_auc_score(sub.failure, sub.verif_mean) if sub.failure.nunique() == 2 else np.nan
    except Exception:
        task_auc = np.nan
    rows.append({
        "task": t,
        "n_succ": len(s), "n_fail": len(f),
        "succ_mean_ent": s.mean() if len(s) else np.nan,
        "fail_mean_ent": f.mean() if len(f) else np.nan,
        "gap": (f.mean() - s.mean()) if len(s) and len(f) else np.nan,
        "task_auc": task_auc,
    })
task_tab = pd.DataFrame(rows).sort_values("gap", ascending=False)

fig, (axA, axB) = plt.subplots(1, 2, figsize=(14, max(3, 0.35 * n_tasks)))

# A) gap bar
colors = ["#2ca02c" if g > 0 else "#d62728" for g in task_tab["gap"].fillna(0)]
axA.barh(task_tab["task"], task_tab["gap"].fillna(0), color=colors,
         edgecolor="black", linewidth=0.5)
axA.axvline(0, color="black", lw=0.8)
axA.set_xlabel("mean entropy gap  (failure − success)")
axA.set_title("Per-task entropy gap  (green = entropy higher on failures)")
axA.invert_yaxis()
for i, (_, r) in enumerate(task_tab.iterrows()):
    axA.annotate(f"{r['n_succ']}/{r['n_succ']+r['n_fail']} succ",
                 (r["gap"] if pd.notna(r["gap"]) else 0, i),
                 textcoords="offset points", xytext=(5, 0), fontsize=8, va="center")

# B) per-task AUC
axB.barh(task_tab["task"], task_tab["task_auc"].fillna(0.5),
         color=["#2ca02c" if (a or 0.5) > 0.5 else "#d62728"
                for a in task_tab["task_auc"].fillna(0.5)],
         edgecolor="black", linewidth=0.5)
axB.axvline(0.5, color="black", lw=0.8, ls="--")
axB.set_xlim(0, 1)
axB.set_xlabel("Per-task ROC-AUC (entropy → failure)")
axB.set_title("Per-task discrimination")
axB.invert_yaxis()

plt.tight_layout()
plt.show()

display(Markdown("#### Per-task summary"))
display(
    task_tab.style.format({
        "succ_mean_ent": "{:.4f}", "fail_mean_ent": "{:.4f}",
        "gap": "{:+.4f}", "task_auc": "{:.3f}",
    }).background_gradient(subset=["gap"], cmap="RdYlGn", vmin=-0.05, vmax=0.05)
      .background_gradient(subset=["task_auc"], cmap="RdYlGn", vmin=0.3, vmax=0.7)
      .hide(axis="index")
)


## Within-task normalised entropy

Tasks have very different baseline entropy levels (boilWater ≈ 0,
toastBread ≈ 0.1). Pooling raw values lets high-baseline tasks dominate
the distribution.

We z-score `verif_mean` within each task, then re-run the success-vs-failure
test. If entropy is a real hardness signal, the effect should **strengthen**
after within-task normalisation (noise from task-baseline variance is removed).


In [ ]:
# ── Within-task z-score analysis ─────────────────────────────────────────────
zsucc = ep_analysis.loc[ep_analysis.coplan_success, "verif_mean_z"].dropna().values
zfail = ep_analysis.loc[~ep_analysis.coplan_success, "verif_mean_z"].dropna().values

u_z, p_z = scipy_stats.mannwhitneyu(zsucc, zfail, alternative="two-sided")
r_z, pp_z = scipy_stats.pointbiserialr(
    ep_analysis.dropna(subset=["verif_mean_z"]).failure,
    ep_analysis.dropna(subset=["verif_mean_z"]).verif_mean_z,
)
auc_z = roc_auc_score(
    ep_analysis.dropna(subset=["verif_mean_z"]).failure,
    ep_analysis.dropna(subset=["verif_mean_z"]).verif_mean_z,
)

fig, ax = plt.subplots(figsize=(8, 5))
parts = ax.violinplot([zsucc, zfail], showmedians=True, widths=0.8)
for i, pc in enumerate(parts["bodies"]):
    pc.set_facecolor(["#4caf50", "#e53935"][i])
    pc.set_alpha(0.55); pc.set_edgecolor("black")
ax.axhline(0, color="grey", ls="--", lw=1)
ax.set_xticks([1, 2])
ax.set_xticklabels([f"Success (n={len(zsucc)})", f"Failure (n={len(zfail)})"])
ax.set_ylabel("Within-task z-score of mean entropy")
ax.set_title(f"Within-task normalised entropy\n"
             f"Mann-Whitney p={p_z:.4f}   point-biserial r={r_z:+.3f}   "
             f"ROC-AUC={auc_z:.3f}")
plt.tight_layout()
plt.show()

print(f"Raw mean-entropy:         r={scipy_stats.pointbiserialr(ep_analysis.failure, ep_analysis.verif_mean)[0]:+.4f}, AUC={roc_auc_score(ep_analysis.failure, ep_analysis.verif_mean):.4f}")
print(f"Within-task-z mean-entropy: r={r_z:+.4f}, AUC={auc_z:.4f}")


## Predictor comparison

A single "mean" is one of several ways to summarise an episode's entropy.
Below we compare each candidate predictor by ROC-AUC on `coplan_failure`.
**High-entropy step count** (number of steps with entropy > threshold) may
outperform the mean because a single confident-but-wrong decision is
often what dooms an episode.


In [ ]:
# ── AUC comparison across candidate predictors ───────────────────────────────
predictors = [
    ("verif_mean",       "verifier mean"),
    ("verif_median",     "verifier median"),
    ("verif_max",        "verifier max"),
    ("verif_high_count", f"verifier high-count  (>{HIGH_ENT_THRESHOLD})"),
    ("verif_last",       "verifier last-step"),
    ("verif_mean_z",     "verifier mean (within-task z)"),
    ("verif_max_z",      "verifier max  (within-task z)"),
    ("plan_mean",        "proposer mean"),
    ("plan_max",         "proposer max"),
    ("num_steps",        "num correction steps"),
]

rows = []
for col, label in predictors:
    if col not in ep_analysis.columns:
        continue
    df = ep_analysis.dropna(subset=[col])
    if df.failure.nunique() != 2 or len(df) < 10:
        continue
    auc = roc_auc_score(df.failure, df[col])
    ap  = average_precision_score(df.failure, df[col])
    r, p = scipy_stats.pointbiserialr(df.failure, df[col])
    rows.append({"predictor": label, "n": len(df), "AUC": auc, "PR-AUC": ap,
                 "point_biserial_r": r, "p": p})

pred_tab = pd.DataFrame(rows).sort_values("AUC", ascending=False)

fig, ax = plt.subplots(figsize=(10, 0.4 * len(pred_tab) + 1))
colors = ["#2ca02c" if a > 0.6 else "#ff9800" if a > 0.55 else "#d62728"
          for a in pred_tab["AUC"]]
ax.barh(pred_tab["predictor"], pred_tab["AUC"], color=colors,
        edgecolor="black", linewidth=0.5)
ax.axvline(0.5, color="grey", ls="--", lw=1, label="chance")
ax.axvline(0.7, color="green", ls="--", lw=1, label="usable threshold")
ax.set_xlim(0.3, 0.8)
ax.set_xlabel("ROC-AUC (predictor → co-plan failure)")
ax.set_title("Predictor comparison")
ax.invert_yaxis()
ax.legend(loc="lower right", fontsize=8)
for i, (_, r) in enumerate(pred_tab.iterrows()):
    ax.annotate(f"{r['AUC']:.3f}", (r["AUC"], i),
                textcoords="offset points", xytext=(4, 0), fontsize=8, va="center")
plt.tight_layout()
plt.show()

display(pred_tab.style.format({"AUC": "{:.3f}", "PR-AUC": "{:.3f}",
                               "point_biserial_r": "{:+.3f}", "p": "{:.4f}"})
        .background_gradient(subset=["AUC"], cmap="RdYlGn", vmin=0.4, vmax=0.7)
        .hide(axis="index"))


## Entropy vs continuous hardness (num correction steps)

Co-plan success is binary. A more graded notion of "hardness" is the
number of correction steps the replanner took: successful episodes that
took many steps were borderline, and failures that took many steps were
severe. If entropy tracks hardness generally, we should see a positive
trend between `num_steps` and `verif_mean` beyond just the binary gap.


In [ ]:
# ── Entropy vs num correction steps ──────────────────────────────────────────
sub = ep_analysis.dropna(subset=["num_steps", "verif_mean"]).copy()
sub["log_ent"] = _log(sub["verif_mean"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Scatter with regression
ax = axes[0]
for ok, color, label in [(True, "#4caf50", "coplan success"),
                         (False, "#e53935", "coplan failure")]:
    s = sub[sub.coplan_success == ok]
    ax.scatter(s.num_steps, s.log_ent, c=color, alpha=0.6, s=50,
               edgecolor="black", lw=0.4, label=label)
# Regression line (all points)
if len(sub) >= 3:
    slope, intercept, r_val, p_val, _ = scipy_stats.linregress(sub.num_steps, sub.log_ent)
    xs = np.linspace(sub.num_steps.min(), sub.num_steps.max(), 50)
    ax.plot(xs, slope * xs + intercept, color="black", lw=1.5, ls="--",
            label=f"fit  r={r_val:+.3f}  p={p_val:.3f}")
ax.set_xlabel("num correction steps")
ax.set_ylabel("log10(mean verifier entropy + 1e-6)")
ax.set_title("Entropy vs graded hardness")
ax.legend(loc="lower right", fontsize=9)

# (b) Mean entropy by num_steps bucket
ax = axes[1]
sub["step_bucket"] = pd.cut(sub.num_steps,
                            bins=[-1, 0, 2, 4, 6, 8, 100],
                            labels=["0", "1-2", "3-4", "5-6", "7-8", "9+"])
buck = sub.groupby("step_bucket", observed=True)["verif_mean"].agg(["mean", "median", "count"]).reset_index()
ax.bar(buck["step_bucket"].astype(str), buck["mean"],
       color="#1f77b4", edgecolor="black", lw=0.5, alpha=0.85)
for i, r in buck.iterrows():
    ax.annotate(f"n={int(r['count'])}", (i, r["mean"]),
                textcoords="offset points", xytext=(0, 3), ha="center", fontsize=8)
ax.set_yscale("log")
ax.set_xlabel("num correction steps")
ax.set_ylabel("mean verifier entropy (log)")
ax.set_title("Mean entropy by step-count bucket")

plt.tight_layout()
plt.show()


## Verifier vs proposer entropy

Two entropy streams come from different LLM calls:

| stream | call | `reasoning_effort` | purpose |
|---|---|---|---|
| **verifier** | subgoal "did we achieve X?" | `"none"` | binary yes/no per step |
| **proposer** | A/B/C/D/E candidate ranking | `"none"` | pick an action from 4+ options |

Both are set to `reasoning_effort="none"` (fast scoring). We check whether
either stream has useful discrimination - if the proposer is essentially
always confident (distribution collapsed to ~0), it is not a usable signal
regardless of the verifier result.


In [ ]:
# ── Verifier vs Proposer entropy distributions ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (prefix, title) in zip(axes, [("verif", "Verifier"), ("plan", "Proposer")]):
    col = f"{prefix}_mean"
    if col not in ep_analysis.columns or ep_analysis[col].notna().sum() < 5:
        ax.set_title(f"{title}: insufficient data")
        ax.axis("off")
        continue
    sub = ep_analysis.dropna(subset=[col])
    s = sub.loc[sub.coplan_success, col].values
    f = sub.loc[~sub.coplan_success, col].values

    parts = ax.violinplot([_log(s), _log(f)], showmedians=True, widths=0.8)
    for i, pc in enumerate(parts["bodies"]):
        pc.set_facecolor(["#4caf50", "#e53935"][i])
        pc.set_alpha(0.55); pc.set_edgecolor("black")
    try:
        u, pp = scipy_stats.mannwhitneyu(s, f, alternative="two-sided")
        auc = roc_auc_score(sub.failure, sub[col])
        subtitle = f"MW p={pp:.4f}  AUC={auc:.3f}"
    except Exception:
        subtitle = ""
    ax.set_xticks([1, 2]); ax.set_xticklabels(["success", "failure"])
    ax.set_ylabel(f"log10({prefix} mean entropy + 1e-6)")
    ax.set_title(f"{title} entropy: success vs failure\n{subtitle}")

plt.tight_layout()
plt.show()

# ── How often is each stream active / measurable? ───────────────────────────
diag_rows = []
for prefix in ["verif", "plan"]:
    col_mean = f"{prefix}_mean"
    col_n    = f"{prefix}_n_steps"
    if col_mean in ep_analysis.columns:
        has = ep_analysis[col_mean].notna().sum()
        med_n = ep_analysis[col_n].median() if col_n in ep_analysis.columns else np.nan
        med_ent = ep_analysis[col_mean].median()
        max_ent = ep_analysis[col_mean].max()
        diag_rows.append({
            "stream": prefix, "episodes_with_data": has,
            "median_steps_per_ep": med_n,
            "median_ent": med_ent, "max_ent": max_ent,
        })
display(pd.DataFrame(diag_rows).style.format(precision=5).hide(axis="index"))


## Error taxonomy vs entropy  (run 4)

**Hypothesis from physical intuition**: the LLM can only be torn between
options if it actually *sees* the relevant evidence. So:

- **Perception / context errors** (`PER-*`, `CTX-HIDDEN`) - the LLM has
  the wrong or missing information → it should be **confidently wrong**
  → **low entropy**.
- **Reasoning errors** (`RSN-*`) - the LLM sees the situation but makes
  a bad inference → it may have been genuinely torn → **higher entropy**.
- **Execution / affordance** (`EXEC-SIM`, `CTX-AFFORD`) - mixed: some
  surface as detectable inconsistencies, some don't.

If this hypothesis holds, entropy is useful for triaging **reasoning**
failures but blind to **perception** failures - which is important to
know when deciding where entropy-gated recovery is safe to deploy.

The taxonomy (run 4) was recorded on a previous pipeline version. It
covers 52 episodes; 41 of those also failed in the current sim-grounded
run (joined here). Episodes that recovered between runs are listed
separately in the diagnostic output.


In [ ]:
# ── Load run-4 error taxonomy and join to current episodes ───────────────────
from reflect.core.paths import sim_results_root

TAXONOMY_PATH = sim_results_root() / "error_taxonomy_all_run4.csv"

if not TAXONOMY_PATH.exists():
    display(Markdown(f"_Taxonomy file not found: {TAXONOMY_PATH}_"))
    taxonomy = pd.DataFrame()
else:
    taxonomy = pd.read_csv(TAXONOMY_PATH)
    taxonomy = taxonomy[["task", "episode", "taxonomy_code",
                         "r_gt_error_type", "r_error_type",
                         "Loc", "Exp"]].copy()
    # Strip trailing hyphenated discriminator to match ep_analysis format
    taxonomy["key"] = taxonomy["task"] + "/" + taxonomy["episode"]

    # Join on (task, episode)
    ep_tax = ep_analysis.merge(taxonomy, on=["task", "episode"], how="left",
                               suffixes=("", "_tax"))
    ep_tax["has_taxonomy"] = ep_tax["taxonomy_code"].notna()

    n_total = len(taxonomy)
    n_join_fail = (ep_tax["has_taxonomy"] & ~ep_tax["coplan_success"]).sum()
    n_join_succ = (ep_tax["has_taxonomy"] &  ep_tax["coplan_success"]).sum()
    n_missing  = (~ep_analysis.set_index(["task", "episode"]).index.isin(
        taxonomy.set_index(["task", "episode"]).index)
        & ~ep_analysis["coplan_success"]
    ).sum()

    display(Markdown(
        f"**Taxonomy coverage**: {n_total} labelled episodes · "
        f"{n_join_fail} still fail in current run · "
        f"{n_join_succ} now succeed (recovered by sim-grounded replan) · "
        f"{n_missing} current failures lack a taxonomy label"
    ))

    # ── Summary table by taxonomy code ──────────────────────────────────────
    tax_group = (
        ep_tax[ep_tax.has_taxonomy]
        .groupby("taxonomy_code")
        .agg(n=("verif_mean", "size"),
             n_fail_now=("failure", "sum"),
             recovery_rate=("coplan_success", "mean"),
             verif_mean=("verif_mean", "mean"),
             verif_median=("verif_mean", "median"),
             verif_max_mean=("verif_max", "mean"),
             verif_high_count_mean=("verif_high_count", "mean"))
        .reset_index()
        .sort_values("verif_mean", ascending=False)
    )
    display(Markdown("#### Per-code summary"))
    display(tax_group.style.format({
        "recovery_rate": "{:.0%}",
        "verif_mean": "{:.4f}", "verif_median": "{:.4f}",
        "verif_max_mean": "{:.4f}", "verif_high_count_mean": "{:.2f}",
    }).background_gradient(subset=["verif_mean"], cmap="YlOrRd")
      .hide(axis="index"))

    # ── Violin per taxonomy code ────────────────────────────────────────────
    # Higher-level grouping aligned with the hypothesis
    code_family = {
        "PER-SPATIAL": "PER  (perception)",
        "PER-AUDIO":   "PER  (perception)",
        "CTX-HIDDEN":  "CTX  (hidden/affordance)",
        "CTX-AFFORD":  "CTX  (hidden/affordance)",
        "RSN-PLAN":    "RSN  (reasoning)",
        "RSN-DIAG":    "RSN  (reasoning)",
        "EXEC-SIM":    "EXEC (sim/physics)",
        "TR-EMBED":    "TR   (translation)",
    }
    ep_tax["family"] = ep_tax["taxonomy_code"].map(code_family)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # (a) by fine-grained code
    ax = axes[0]
    tax_order = tax_group["taxonomy_code"].tolist()
    data = []
    labels = []
    for code in tax_order:
        vals = ep_tax.loc[ep_tax.taxonomy_code == code, "verif_mean"].dropna().values
        if len(vals):
            data.append(_log(vals))
            labels.append(f"{code}\n(n={len(vals)})")
    if data:
        parts = ax.violinplot(data, showmedians=True, widths=0.85)
        for pc in parts["bodies"]:
            pc.set_facecolor("#6baed6"); pc.set_alpha(0.6); pc.set_edgecolor("black")
        ax.set_xticks(range(1, len(data) + 1))
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_ylabel("log10(mean verifier entropy + 1e-6)")
        ax.set_title("Verifier entropy by error-taxonomy code")

    # (b) by higher-level family (cleaner test of the hypothesis)
    ax = axes[1]
    fam_order = ["PER  (perception)", "CTX  (hidden/affordance)",
                 "EXEC (sim/physics)", "RSN  (reasoning)", "TR   (translation)"]
    fam_data = []; fam_labels = []; fam_colors = []
    palette = {"PER  (perception)":       "#d62728",
               "CTX  (hidden/affordance)": "#ff7f0e",
               "EXEC (sim/physics)":       "#9467bd",
               "RSN  (reasoning)":         "#2ca02c",
               "TR   (translation)":       "#8c564b"}
    for fam in fam_order:
        vals = ep_tax.loc[ep_tax.family == fam, "verif_mean"].dropna().values
        if len(vals):
            fam_data.append(_log(vals))
            fam_labels.append(f"{fam}\n(n={len(vals)})")
            fam_colors.append(palette[fam])
    if fam_data:
        parts = ax.violinplot(fam_data, showmedians=True, widths=0.85)
        for pc, color in zip(parts["bodies"], fam_colors):
            pc.set_facecolor(color); pc.set_alpha(0.55); pc.set_edgecolor("black")
        ax.set_xticks(range(1, len(fam_data) + 1))
        ax.set_xticklabels(fam_labels, fontsize=9)
        ax.set_ylabel("log10(mean verifier entropy + 1e-6)")
        ax.set_title("Verifier entropy by error family  (hypothesis view)")

    plt.tight_layout()
    plt.show()

    # ── Formal test: is RSN entropy > PER/CTX entropy? ──────────────────────
    rsn = ep_tax.loc[ep_tax.family == "RSN  (reasoning)", "verif_mean"].dropna()
    blind = ep_tax.loc[ep_tax.family.isin(
        ["PER  (perception)", "CTX  (hidden/affordance)"]), "verif_mean"].dropna()
    if len(rsn) > 2 and len(blind) > 2:
        u, pval = scipy_stats.mannwhitneyu(rsn, blind, alternative="greater")
        display(Markdown(
            f"**Hypothesis test** (one-sided): `RSN > PER/CTX` entropy.  "
            f"n(RSN)={len(rsn)}, n(PER+CTX)={len(blind)}.  "
            f"median RSN={rsn.median():.4f} vs PER/CTX={blind.median():.4f}.  "
            f"Mann-Whitney U={u:.0f}, **p = {pval:.4f}**"
        ))

    # Keep for summary cell
    ep_tax_global = ep_tax


## Episode trace navigator

Per-episode drill-down. The per-step entropy bar chart now uses a **log-y**
axis and marks the high-entropy threshold so that spikes are visible even
when the rest of the trace is near zero.


In [ ]:
def render_episode_trace(task_name: str, episode_name: str):
    """Render a detailed trace for one episode."""
    key = f"{task_name}/{episode_name}"
    trace = trace_jsons.get(key)
    ep_det = detector_trace[(detector_trace["task"] == task_name) & (detector_trace["episode"] == episode_name)]
    ep_plan = plan_trace[(plan_trace["task"] == task_name) & (plan_trace["episode"] == episode_name)] if not plan_trace.empty else pd.DataFrame()
    ep_meta = episode_metrics[(episode_metrics["task"] == task_name) & (episode_metrics["episode"] == episode_name)]

    if not ep_meta.empty:
        row = ep_meta.iloc[0]
        display(Markdown(
            f"### {key}\n"
            f"**Status**: {row.get('status')} · "
            f"**Co-plan**: {'✓' if row.get('coplan_success') else '✗'} · "
            f"**GT failure**: {row.get('gt_failure_reason', '-')} at {row.get('gt_failure_step', '-')} · "
            f"**Predicted**: {str(row.get('pred_failure_reason', '-'))[:80]}... at {row.get('pred_failure_step', '-')}"
        ))

    if not ep_det.empty:
        display(Markdown("#### Subgoal verification"))
        cols = ["step", "subgoal", "predicted_success", "oracle_success",
                "entropy", "confidence", "score_status"]
        avail = [c for c in cols if c in ep_det.columns]
        display(ep_det[avail].style.format(precision=5).background_gradient(
            subset=["entropy"] if "entropy" in avail else [], cmap="YlOrRd"
        ).hide(axis="index"))

        # Log-scale step-by-step entropy
        fig, ax = plt.subplots(figsize=(10, 3))
        steps = range(len(ep_det))
        ents = ep_det["entropy"].fillna(0).values + 1e-6  # log floor
        colors = ["#e53935" if (s is False or (isinstance(s, (bool, np.bool_)) and not s))
                  else "#4caf50"
                  for s in ep_det["oracle_success"].values]
        ax.bar(steps, ents, color=colors, edgecolor="black", linewidth=0.5, alpha=0.85)
        ax.axhline(HIGH_ENT_THRESHOLD, color="black", ls="--", lw=0.8,
                   label=f"high-entropy threshold ({HIGH_ENT_THRESHOLD})")
        ax.set_yscale("log")
        ax.set_xticks(list(steps))
        ax.set_xticklabels(ep_det["step"].astype(str).values, fontsize=7,
                           rotation=45, ha="right")
        ax.set_ylabel("entropy (log)")
        ax.set_title(f"Verification entropy per step - {key}")
        ax.legend(fontsize=8, loc="upper right")
        plt.tight_layout()
        plt.show()

    if not ep_plan.empty:
        display(Markdown("#### Plan step selection"))
        pcols = ["step_index", "selected_action", "confidence", "entropy", "score_status"]
        pavail = [c for c in pcols if c in ep_plan.columns]
        display(ep_plan[pavail].style.format(precision=5).background_gradient(
            subset=["entropy"] if "entropy" in pavail else [], cmap="YlOrRd"
        ).hide(axis="index"))

    if trace:
        prompts_log = trace.get("prompts_log", [])
        if prompts_log:
            display(Markdown("#### LLM call log"))
            for i, entry in enumerate(prompts_log):
                call_type = entry.get("call", "?")
                prompt_user = entry.get("prompt", {}).get("user", "")[:300]
                response = str(entry.get("response", ""))[:300]
                score = entry.get("score") or {}
                ent = score.get("entropy", "-")
                conf = score.get("confidence", "-")
                display(Markdown(
                    f"**{i+1}. `{call_type}`** - entropy={ent}, conf={conf}\n\n"
                    f"> *Prompt (trunc)*: {prompt_user}...\n\n"
                    f"> *Response (trunc)*: {response}..."
                ))

print("Trace navigator ready.")


In [ ]:
# ── Interactive episode selector ──────────────────────────────────────────────
try:
    import ipywidgets as widgets

    episode_keys = [f"{r.task}/{r.episode}" for _, r in episode_metrics.iterrows()]
    dropdown = widgets.Dropdown(options=episode_keys, description="Episode:")
    output = widgets.Output()

    def _on_select(change):
        output.clear_output(wait=True)
        with output:
            t, e = change["new"].split("/", 1)
            render_episode_trace(t, e)

    dropdown.observe(_on_select, names="value")
    display(dropdown, output)
    if episode_keys:
        _on_select({"new": episode_keys[0]})
except ImportError:
    display(Markdown("_ipywidgets not available - showing first 3 episodes._"))
    for _, row in episode_metrics.head(3).iterrows():
        render_episode_trace(row["task"], row["episode"])


## Failure localisation vs entropy at the GT failure step

An episode's failure is **correctly localised** when the LLM's predicted
failure step matches the ground-truth step. `gt_failure_step` is sometimes
a single timestamp like `"00:36"` and sometimes a list of acceptable
timestamps like `"['00:25', '00:26', ...]"` (the failure extends over a
window). We parse both and count a match if the predicted step falls
inside the GT window.

We then check entropy **at the GT failure step** - is it elevated when
the LLM correctly localises the failure?


In [ ]:
# ── Parse list-style step values and compute localisation ───────────────────
import ast

def _parse_steps(val):
    """Return a set of step strings from a cell that may contain a scalar or list repr."""
    if val is None:
        return set()
    try:
        if pd.isna(val):
            return set()
    except (ValueError, TypeError):
        pass
    s = str(val).strip()
    if not s:
        return set()
    if s.startswith("["):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set)):
                return {str(x).strip() for x in parsed if str(x).strip()}
        except (ValueError, SyntaxError):
            pass
    return {s.strip("[]'\"")}

loc_rows = []
for _, row in episode_metrics.iterrows():
    pred_set = _parse_steps(row.get("pred_failure_step"))
    gt_set   = _parse_steps(row.get("gt_failure_step"))
    if not gt_set:
        continue
    # Localisation = any predicted step falls in the GT window
    localised = bool(pred_set & gt_set) if pred_set else False
    loc_rows.append({
        "task": row["task"],
        "episode": row["episode"],
        "pred_steps": ",".join(sorted(pred_set)) if pred_set else "",
        "gt_steps":   ",".join(sorted(gt_set)),
        "gt_first":   sorted(gt_set)[0],
        "localised":  localised,
    })
loc_df = pd.DataFrame(loc_rows)

if loc_df.empty:
    display(Markdown("_No episodes with ground-truth failure steps._"))
else:
    n_loc = int(loc_df["localised"].sum())
    n_total = len(loc_df)
    display(Markdown(
        f"**Localisation accuracy**: {n_loc}/{n_total} ({100 * n_loc / n_total:.1f}%)"
    ))

    # Entropy at the first GT failure step (step strings like "00:36")
    gt_step_entropy = []
    for _, r in loc_df.iterrows():
        det_ep = detector_trace[
            (detector_trace["task"] == r["task"]) &
            (detector_trace["episode"] == r["episode"])
        ]
        hit = det_ep[det_ep["step"].astype(str) == r["gt_first"]]
        gt_step_entropy.append(hit["entropy"].iloc[0] if not hit.empty else np.nan)
    loc_df["gt_step_entropy"] = gt_step_entropy

    # Join with episode mean/max entropy
    loc_merged = loc_df.merge(
        ep_analysis[["task", "episode", "verif_mean", "verif_max", "coplan_success"]],
        on=["task", "episode"], how="left",
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # (a) Entropy at GT step, localised vs not  (log)
    for ax_idx, (col, label) in enumerate([
        ("gt_step_entropy", "entropy AT GT failure step"),
        ("verif_mean",      "episode mean entropy"),
    ]):
        ax = axes[ax_idx]
        sub = loc_merged.dropna(subset=[col])
        A = sub.loc[sub.localised, col].values
        B = sub.loc[~sub.localised, col].values
        if len(A) and len(B):
            parts = ax.violinplot([_log(A), _log(B)], showmedians=True, widths=0.8)
            for i, pc in enumerate(parts["bodies"]):
                pc.set_facecolor(["#1f77b4", "#d62728"][i])
                pc.set_alpha(0.55); pc.set_edgecolor("black")
            u, p_mw = scipy_stats.mannwhitneyu(A, B, alternative="two-sided")
            ax.set_title(f"{label}\nlocalised vs NOT  (MW p={p_mw:.4f})")
        ax.set_xticks([1, 2])
        ax.set_xticklabels([f"localised (n={len(A)})", f"NOT localised (n={len(B)})"])
        ax.set_ylabel(f"log10({col} + 1e-6)")

    plt.tight_layout()
    plt.show()

    display(Markdown("#### Per-episode detail (sorted by entropy at GT step)"))
    show_cols = ["task", "episode", "localised", "gt_steps", "pred_steps",
                 "gt_step_entropy", "verif_mean", "verif_max", "coplan_success"]
    display(
        loc_merged[[c for c in show_cols if c in loc_merged.columns]]
        .sort_values("gt_step_entropy", ascending=False)
        .style.format({"gt_step_entropy": "{:.5f}",
                       "verif_mean": "{:.5f}",
                       "verif_max": "{:.5f}"})
        .background_gradient(subset=["gt_step_entropy"], cmap="YlOrRd")
        .hide(axis="index")
    )


## Summary report


In [ ]:
# ── Aggregate summary with new predictors + taxonomy ─────────────────────────
n_episodes  = len(episode_metrics)
n_completed = (episode_metrics["status"] == "ok").sum()
n_coplan    = episode_metrics["coplan_success"].fillna(False).sum()
coplan_rate = n_coplan / n_completed if n_completed else 0

det_scored  = detector_trace.dropna(subset=["entropy"])
plan_scored = plan_trace.dropna(subset=["entropy"]) if not plan_trace.empty else pd.DataFrame()

auc_mean    = roc_auc_score(ep_analysis.failure, ep_analysis.verif_mean)
auc_max     = roc_auc_score(ep_analysis.failure, ep_analysis.verif_max.fillna(0))
auc_hc      = roc_auc_score(ep_analysis.failure, ep_analysis.verif_high_count.fillna(0))
if ep_analysis["verif_mean_z"].notna().sum() > 10:
    dfz = ep_analysis.dropna(subset=["verif_mean_z"])
    auc_z = roc_auc_score(dfz.failure, dfz.verif_mean_z)
else:
    auc_z = float("nan")

loc_acc = loc_df["localised"].mean() if not loc_df.empty else float("nan")

tax_line = ""
try:
    if not taxonomy.empty and 'ep_tax_global' in globals():
        rsn = ep_tax_global.loc[ep_tax_global.family == "RSN  (reasoning)", "verif_mean"].dropna()
        blind = ep_tax_global.loc[ep_tax_global.family.isin(
            ["PER  (perception)", "CTX  (hidden/affordance)"]), "verif_mean"].dropna()
        if len(rsn) > 2 and len(blind) > 2:
            u, pv = scipy_stats.mannwhitneyu(rsn, blind, alternative="greater")
            tax_line = (f"| RSN vs PER/CTX (one-sided MW) | "
                        f"median RSN={rsn.median():.4f} vs PER/CTX={blind.median():.4f}, "
                        f"p={pv:.4f} |")
except Exception:
    pass

summary_md = f"""
| Metric | Value |
|--------|-------|
| Episodes run | {n_episodes} |
| Completed (status=ok) | {n_completed} |
| Co-plan success rate | {int(n_coplan)}/{n_completed} ({100*coplan_rate:.1f}%) |
| Mean verifier entropy (all steps) | {det_scored['entropy'].mean():.5f} |
| Median verifier entropy | {det_scored['entropy'].median():.5f} |
| Mean proposer entropy | {plan_scored['entropy'].mean() if not plan_scored.empty else float('nan'):.5f} |
| ROC-AUC verifier mean → failure | **{auc_mean:.3f}** |
| ROC-AUC verifier max → failure | {auc_max:.3f} |
| ROC-AUC verifier high-count → failure | {auc_hc:.3f} |
| ROC-AUC within-task-z mean → failure | {auc_z:.3f} |
| Failure localisation accuracy | {100*loc_acc:.1f}% |
{tax_line}
| Model | {MODEL} (gen={GEN_REASONING_EFFORT}, score={SCORE_REASONING_EFFORT}) |
"""
display(Markdown(summary_md))
